# LLM Fine-Tuning Toolkit — Quickstart

This notebook demonstrates the full fine-tuning pipeline:
1. Loading and preparing a dataset
2. Configuring QLoRA parameters
3. Launching training
4. Evaluating the fine-tuned model
5. Running inference

**Prerequisites:** Start the API server first:
```bash
uvicorn app.main:app --reload --port 8003
```

In [ ]:
import requests
import json
import time

BASE_URL = "http://localhost:8003"

def api(method, path, **kwargs):
    resp = getattr(requests, method)(f"{BASE_URL}{path}", **kwargs)
    resp.raise_for_status()
    return resp.json()

# Health check
health = api("get", "/health")
print(json.dumps(health, indent=2))

## 1. Dataset Preparation

Load a dataset, auto-detect format, apply quality filters, and split into train/val.

In [ ]:
# Prepare dataset from HuggingFace Hub
dataset_config = {
    "source": "tatsu-lab/alpaca",
    "source_format": "auto",
    "target_format": "alpaca",
    "split_ratio": 0.9,
    "name": "alpaca_demo",
    "max_samples": 1000,
    "filters": {
        "min_length": 20,
        "max_length": 4096,
        "remove_duplicates": True
    }
}

dataset_info = api("post", "/datasets/prepare", json=dataset_config)
print(f"Dataset: {dataset_info['name']}")
print(f"Samples: {dataset_info['num_samples']}")
print(f"Split: train={dataset_info['split_sizes']['train']}, val={dataset_info['split_sizes']['val']}")

In [ ]:
# List all prepared datasets
datasets = api("get", "/datasets")
for ds in datasets:
    print(f"  {ds['name']}: {ds['num_samples']} samples ({ds['format']})")

## 2. Configure & Launch Training

Fine-tune with QLoRA using the `balanced` preset (rank=16, alpha=32).

In [ ]:
# Launch SFT training
training_config = {
    "dataset_name": "alpaca_demo",
    "base_model": "mistralai/Mistral-7B-v0.3",
    "lora_config": {
        "preset": "balanced",
        "quantization_bits": 4
    },
    "training_args": {
        "num_epochs": 1,
        "learning_rate": 2e-4,
        "per_device_batch_size": 4,
        "gradient_accumulation_steps": 4,
        "logging_steps": 5
    },
    "use_unsloth": True
}

job = api("post", "/training/sft", json=training_config)
job_id = job["job_id"]
print(f"Job ID: {job_id}")
print(f"Status: {job['status']}")

if job["status"] == "failed":
    print(f"Error: {job.get('error_message', 'Unknown')}")

In [ ]:
# Monitor training progress
if job["status"] != "failed":
    while True:
        status = api("get", f"/training/status/{job_id}")
        print(f"Progress: {status['progress']:.1f}% | "
              f"Loss: {status['metrics'].get('train_loss', 'N/A')} | "
              f"Status: {status['status']}")
        if status["status"] in ("completed", "failed"):
            break
        time.sleep(10)
    
    if status["status"] == "completed":
        print(f"\nAdapter saved to: {status['adapter_path']}")

## 3. Evaluate

Compute perplexity, BLEU, and ROUGE on the validation set.

In [ ]:
# Evaluate the fine-tuned adapter
# Replace adapter_path with the actual path from training
adapter_path = status.get("adapter_path", "./outputs/sft_job_xxx/final_adapter")

eval_config = {
    "adapter_path": adapter_path,
    "dataset_name": "alpaca_demo",
    "metrics": ["perplexity", "bleu", "rouge"],
    "num_samples": 50
}

result = api("post", "/evaluate", json=eval_config)
print(f"Perplexity: {result.get('perplexity', 'N/A')}")
print(f"BLEU: {result.get('bleu', 'N/A')}")
print(f"ROUGE: {json.dumps(result.get('rouge', {}), indent=2)}")

print("\nSample outputs:")
for s in result.get("samples", [])[:3]:
    print(f"  Prompt: {s['prompt'][:80]}...")
    print(f"  Generated: {s['generated'][:100]}...")
    print()

## 4. Compare Base vs Fine-Tuned

In [ ]:
compare_config = {
    "adapter_path": adapter_path,
    "dataset_name": "alpaca_demo",
    "num_samples": 20
}

comparison = api("post", "/evaluate/compare", json=compare_config)
print("Base model scores:", comparison["base_scores"])
print("Fine-tuned scores:", comparison["finetuned_scores"])
print("Improvement %:", comparison["improvement_pct"])

## 5. Inference

Generate text with the fine-tuned model.

In [ ]:
# Generate text
gen_config = {
    "prompt": "### Instruction:\nExplain the concept of transfer learning in machine learning.\n\n### Response:\n",
    "adapter_path": adapter_path,
    "max_tokens": 256,
    "temperature": 0.7,
    "top_p": 0.9
}

output = api("post", "/inference/generate", json=gen_config)
print(f"Generated ({output['tokens_used']} tokens, {output['generation_time_ms']:.0f}ms):")
print(output["text"])

In [ ]:
# Unload models to free GPU memory
api("post", "/inference/unload")
print("Models unloaded.")

## Model Registry

In [ ]:
# List all registered adapters
models = api("get", "/models")
for m in models:
    print(f"  {m['adapter_name']} | {m['base_model']} | {m['training_type']} | {m.get('file_size_mb', '?')} MB")